In [0]:
# Configurações
catalog = "v_credit_medallion"
silver_schema_name = "silver"
gold_schema_name = "gold"

# Schemas completos
silver_schema = f"{catalog}.{silver_schema_name}"
gold_schema = f"{catalog}.{gold_schema_name}"

janela_recontato_dias = 7
EXPORTAR_CSV = True  # True: exporta os cinco CSVs ao final desta mesma tarefa.
# Preserva o fuso da sessão, como a Silver original.
fuso_horario = spark.conf.get('spark.sql.session.timeZone')


## Funções e contrato de leitura


In [0]:
from functools import reduce
from pyspark.sql import functions as F, Window

# Contrato da Silver: nomes e tipos abaixo; semântica nas células Markdown deste notebook.
CONTRATO = {'tb_chamados': {'id_chamado': 'int',
                 'id_cliente': 'bigint',
                 'id_atendente': 'int',
                 'id_canal': 'int',
                 'resolvido': 'string',
                 'motivo': 'string',
                 'tempo_espera': 'int',
                 'tempo_atendimento': 'int',
                 'flg_abandonado': 'boolean'},
 'tb_chamados_hora': {'id_chamado': 'int',
                      'id_cliente': 'bigint',
                      'hora_abertura_chamado': 'timestamp',
                      'hora_inicio_atendimento': 'timestamp',
                      'hora_finalizacao_atendimento': 'timestamp',
                      'flg_ano_corrigido': 'boolean',
                      'flg_virada_dia': 'boolean'},
 'tb_custos': {'id_custo': 'int', 'id_chamado': 'int', 'custo': 'decimal(10,2)'},
 'tb_pesquisa_satisfacao': {'id_pesquisa': 'int', 'id_chamado': 'int', 'nota_atendimento': 'int'},
 'tb_clientes': {'id_cliente': 'bigint',
                 'email': 'string',
                 'regiao': 'string',
                 'idade': 'int',
                 'data_cadastro': 'date',
                 'dia_vencimento_fatura': 'int',
                 'nome_cliente': 'string'},
 'tb_atendentes': {'id_atendente': 'int',
                   'nome_atendente': 'string',
                   'nivel_atendimento': 'int',
                   'data_admissao': 'date',
                   'data_desligamento': 'date',
                   'status_atendente': 'string'},
 'tb_motivos': {'id_motivo': 'int',
                'nome_motivo': 'string',
                'categoria': 'string',
                'criticidade': 'string'},
 'tb_canais': {'id_canal': 'int',
               'nome_canal': 'string',
               'tipo_canal': 'string',
               'status_canal': 'string'}}
CHAVES = {'tb_chamados':'id_chamado','tb_chamados_hora':'id_chamado',
    'tb_custos':'id_custo','tb_pesquisa_satisfacao':'id_pesquisa','tb_clientes':'id_cliente',
    'tb_atendentes':'id_atendente','tb_motivos':'id_motivo','tb_canais':'id_canal'}


def validar_silver(tabelas):
    """Falha antes da escrita se a Silver não cumpre o contrato. Não limpa dados brutos."""
    for nome, contrato in CONTRATO.items():
        if nome not in tabelas:
            raise ValueError(f'Silver ausente: {nome}. Execute Bronze_to_Silver primeiro.')
        df = tabelas[nome]
        faltantes = set(contrato) - set(df.columns)
        if faltantes:
            raise ValueError(f'{nome}: faltam colunas {sorted(faltantes)}; veja o contrato neste notebook')
        tipos = {f.name:f.dataType.simpleString() for f in df.schema.fields}
        incorretos = {c:(tipos[c],tipo) for c,tipo in contrato.items() if tipos[c] != tipo}
        if incorretos:
            raise ValueError(f'{nome}: tipos (recebido, esperado): {incorretos}')
        # Ignora metadados técnicos da Silver; duplicatas de negócio devem ter sido tratadas lá.
        df = df.select(*contrato)
        chave = CHAVES[nome]
        if df.filter(F.col(chave).isNull() | (F.trim(F.col(chave)) == '')).limit(1).count():
            raise ValueError(f'{nome}: chave {chave} nula/vazia')
        if df.groupBy(chave).count().filter('count > 1').limit(1).count():
            raise ValueError(f'{nome}: chave {chave} duplicada; corrija na Silver')
        tabelas[nome] = df
    # Horas têm no máximo uma linha por chamado; custos e pesquisas são agregados abaixo.
    return tabelas


def construir_gold(tabelas, janela_recontato_dias=7):
    if not isinstance(janela_recontato_dias, int) or janela_recontato_dias < 1:
        raise ValueError('janela_recontato_dias deve ser inteiro positivo')
    t = validar_silver(dict(tabelas))
    t['tb_chamados'] = (t['tb_chamados'].withColumn('resolvido_silver', F.col('resolvido'))
        .withColumn('resolvido',
        F.when(F.col('resolvido') == 'Sim', True).when(F.col('resolvido') == 'Não', False)))
    custo = t['tb_custos'].groupBy('id_chamado').agg(
        F.count('*').alias('qtd_registros_custo'),
        F.sum('custo').alias('custo_total'),
        F.sum(F.col('custo').isNull().cast('int')).alias('qtd_custos_nulos'),
        F.sum((F.col('custo') < 0).cast('int')).alias('qtd_custos_negativos'))
    nota = t['tb_pesquisa_satisfacao'].groupBy('id_chamado').agg(
        F.count('*').alias('qtd_pesquisas'),
        F.count('nota_atendimento').alias('qtd_notas_preenchidas'),
        F.sum(F.col('nota_atendimento').between(1,5).cast('int')).alias('qtd_notas_validas'),
        F.sum((F.col('nota_atendimento').isNotNull() & ~F.col('nota_atendimento').between(1,5)).cast('int')).alias('qtd_notas_invalidas'),
        F.avg(F.when(F.col('nota_atendimento').between(1,5),F.col('nota_atendimento'))).alias('nota_media'))
    hora = t['tb_chamados_hora'].withColumnRenamed('id_cliente','id_cliente_horario').withColumn('_tem_horario',F.lit(True))
    cliente = t['tb_clientes'].withColumnRenamed('email','email_cliente').withColumn('_tem_cliente',F.lit(True))
    atendente = t['tb_atendentes'].withColumn('_tem_atendente',F.lit(True))
    def chave_motivo(coluna):
        return F.regexp_replace(F.translate(F.lower(F.trim(coluna)),
            'áàâãäéèêëíìîïóòôõöúùûüç', 'aaaaaeeeeiiiiooooouuuuc'), r'\s+', ' ')
    motivo = (t['tb_motivos'].withColumn('_chave_motivo', chave_motivo(F.col('nome_motivo')))
        .withColumn('_tem_motivo', F.lit(True)))
    if motivo.filter(F.col('_chave_motivo').isNull() | (F.col('_chave_motivo') == '')).limit(1).count():
        raise ValueError('tb_motivos: nome vazio/nulo impede o mapeamento de id_motivo na Gold')
    if motivo.groupBy('_chave_motivo').count().filter('count > 1').limit(1).count():
        raise ValueError('tb_motivos: nomes normalizados ambíguos; Gold interrompida para não multiplicar chamados')
    canal = t['tb_canais'].withColumn('_tem_canal',F.lit(True))
    g = (t['tb_chamados'].withColumn('_chave_motivo', chave_motivo(F.col('motivo')))
        .withColumnRenamed('motivo', 'motivo_informado_silver')
        .withColumnRenamed('tempo_espera', 'tempo_espera_silver_segundos')
        .withColumnRenamed('tempo_atendimento', 'tempo_atendimento_silver_segundos')
        .withColumnRenamed('flg_abandonado', 'abandonado_silver'))
    for df, chave in [(hora,'id_chamado'),(custo,'id_chamado'),(nota,'id_chamado'),
                      (cliente,'id_cliente'),(atendente,'id_atendente'),(motivo,'_chave_motivo'),(canal,'id_canal')]:
        g = g.join(df,chave,'left')
    # Ausência de custo não vira custo zero. Apenas contagens recebem zero.
    contagens = ['qtd_registros_custo','qtd_custos_nulos','qtd_custos_negativos',
                 'qtd_pesquisas','qtd_notas_preenchidas','qtd_notas_validas','qtd_notas_invalidas']
    g = g.fillna(0, subset=contagens)
    abertura = F.col('hora_abertura_chamado'); inicio = F.col('hora_inicio_atendimento'); fim = F.col('hora_finalizacao_atendimento')
    cronologia = (inicio < abertura) | (fim < abertura) | (fim < inicio)
    g = (g.withColumn('flag_horario_ausente',F.col('_tem_horario').isNull())
        .withColumn('flag_horario_incompleto',abertura.isNull() | fim.isNull())
        .withColumn('flag_cronologia_invalida',F.coalesce(cronologia,F.lit(False)))
        .withColumn('flag_cliente_horario_divergente',F.coalesce(F.col('id_cliente') != F.col('id_cliente_horario'),F.lit(False)))
        .withColumn('flag_cliente_ausente',F.col('_tem_cliente').isNull())
        .withColumn('flag_atendente_ausente',F.coalesce(F.lower(F.col('tipo_canal')) == 'humano', F.lit(False)) & F.col('_tem_atendente').isNull())
        .withColumn('flag_motivo_ausente',F.col('_tem_motivo').isNull())
        .withColumn('flag_canal_ausente',F.col('_tem_canal').isNull())
        .withColumn('flag_resolucao_ausente',F.col('resolvido').isNull())
        .withColumn('flag_custo_ausente',F.col('qtd_registros_custo') == 0)
        .withColumn('flag_custo_invalido',(F.col('qtd_custos_nulos') > 0) | (F.col('qtd_custos_negativos') > 0))
        .withColumn('flag_pesquisa_ausente',F.col('qtd_pesquisas') == 0)
        .withColumn('flag_nota_ausente',(F.col('qtd_pesquisas') > 0) & (F.col('qtd_notas_preenchidas') == 0))
        .withColumn('flag_nota_invalida',F.col('qtd_notas_invalidas') > 0)
        .withColumn('flag_idade_invalida',F.col('idade').isNotNull() & ~F.col('idade').between(18,100))
        .withColumn('flag_idade_ausente',F.col('idade').isNull()))
    humano = F.lower(F.col('tipo_canal')) == 'humano'
    autosservico = F.lower(F.col('tipo_canal')).isin('autosservico', 'autosserviço')
    tempo_valido = ~F.col('flag_horario_incompleto') & ~F.col('flag_cronologia_invalida')
    g = (g.withColumn('abandonado',F.when(tempo_valido & humano,inicio.isNull()))
        .withColumn('tempo_espera_segundos',F.when(tempo_valido & inicio.isNotNull(),inicio.cast('long') - abertura.cast('long')))
        .withColumn('tempo_ate_abandono_segundos',F.when(F.col('abandonado'),fim.cast('long') - abertura.cast('long')))
        .withColumn('tempo_atendimento_segundos',F.when(tempo_valido & inicio.isNotNull(),fim.cast('long') - inicio.cast('long')))
        .withColumn('tempo_total_segundos',F.when(tempo_valido,fim.cast('long') - abertura.cast('long')))
        .withColumn('autosservico',autosservico)
        .withColumn('etapa_canal',F.when(F.upper('nome_canal') == 'URA','URA')
            .when(F.upper('nome_canal') == 'CHATBOT','CHATBOT')
            .when(F.upper('nome_canal') == 'ATENDIMENTO INICIAL','ATENDIMENTO INICIAL')
            .when(F.upper('nome_canal') == 'ATENDIMENTO ESPECIALIZADO','ATENDIMENTO ESPECIALIZADO'))
        .withColumn('qtd_etapas',F.lit(None).cast('int'))
        .withColumn('flag_etapas_nao_observadas',F.lit(True))
        .withColumn('faixa_idade',F.when(F.col('idade').isNull() | F.col('flag_idade_invalida'),'Não informada/inválida')
            .when(F.col('idade') < 18,'00-17').when(F.col('idade') < 30,'18-29')
            .when(F.col('idade') < 45,'30-44').when(F.col('idade') < 60,'45-59').otherwise('60+'))
        .withColumn('data_abertura',F.to_date(abertura))
        .withColumn('ano_mes_abertura',F.date_format(abertura,'yyyy-MM'))
        .withColumn('hora_abertura',F.hour(abertura)))
    # Histórico estritamente anterior: empates de timestamp não inventam uma sequência.
    elegivel = (F.col('id_cliente').isNotNull() & F.col('id_motivo').isNotNull() & abertura.isNotNull()
        & ~F.col('flag_cronologia_invalida') & ~F.col('flag_cliente_horario_divergente'))
    g = g.withColumn('_instante_abertura_us', F.expr('unix_micros(hora_abertura_chamado)'))
    mesmo_instante = Window.partitionBy('id_cliente', 'id_motivo', 'hora_abertura_chamado')
    g = g.withColumn('_contatos_no_instante', F.sum(elegivel.cast('long')).over(mesmo_instante))
    ordem = Window.partitionBy('id_cliente', 'id_motivo').orderBy('_instante_abertura_us')
    historico = ordem.rangeBetween(Window.unboundedPreceding, -1)
    ultimos_7_dias = ordem.rangeBetween(-7 * 86400 * 1000000, -1)
    # Estrutura preserva o canal ausente do contato imediatamente anterior, sem pular para outro.
    contato = F.struct(abertura.alias('abertura'), F.col('id_chamado').alias('id'),
        F.col('nome_canal').alias('canal'), F.col('_tem_canal').alias('canal_encontrado'),
        F.col('_contatos_no_instante').alias('qtd_no_instante'))
    g = (g.withColumn('_contato_anterior', F.max(F.when(elegivel, contato)).over(historico))
        .withColumn('abertura_contato_anterior', F.when(elegivel, F.col('_contato_anterior.abertura')))
        .withColumn('segundos_desde_contato_anterior',
            (F.col('_instante_abertura_us') - F.expr('unix_micros(abertura_contato_anterior)')) / F.lit(1000000.0))
        .withColumn('recontato', F.when(elegivel, F.coalesce(
            F.col('segundos_desde_contato_anterior').between(0, janela_recontato_dias * 86400), F.lit(False))))
        .withColumn('janela_recontato_dias', F.lit(janela_recontato_dias)))

    # 1. Intensidade da recorrência: mesmo cliente e motivo; limite inferior inclusivo.
    # Zero significa nenhum contato anterior observado, não histórico completo garantido.
    g = g.withColumn('qtd_contatos_anteriores_7d', F.when(elegivel,
        F.coalesce(F.sum(elegivel.cast('long')).over(ultimos_7_dias), F.lit(0))))

    # 2. Direção da transição: preserva também repetições como "URA → URA".
    # Se o último instante tem dois chamados, a origem da transição é ambígua e fica nula.
    g = g.withColumn('transicao_canal_recontato',
        F.when(~F.col('recontato'), F.lit('Sem recontato na janela'))
        .when(F.col('recontato') & (F.col('_contato_anterior.qtd_no_instante') == 1)
            & F.col('_tem_canal').isNotNull() & F.col('_contato_anterior.canal_encontrado').isNotNull()
            & F.col('nome_canal').isNotNull() & F.col('_contato_anterior.canal').isNotNull(),
            F.concat(F.col('_contato_anterior.canal'), F.lit(' → '), F.col('nome_canal'))))

    # 3. Faixas descritivas de espera; não são faixas de SLA aprovado.
    espera = F.col('tempo_espera_segundos')
    g = g.withColumn('faixa_espera', F.when(espera == 0, 'Sem espera')
        .when((espera > 0) & (espera <= 60), 'Até 1 min')
        .when((espera > 60) & (espera <= 300), 'Acima de 1 até 5 min')
        .when((espera > 300) & (espera <= 600), 'Acima de 5 até 10 min')
        .when(espera > 600, 'Acima de 10 min'))
    g = g.withColumn('flag_recontato_nao_avaliavel',F.col('recontato').isNull())
    g = (g.withColumnRenamed('flg_ano_corrigido', 'flag_ano_corrigido')
        .withColumnRenamed('flg_virada_dia', 'flag_virada_dia'))
    flags = sorted(c for c in g.columns if c.startswith('flag_'))
    g = g.fillna(False, subset=flags)
    # Sinaliza completude/limitações, não é uma regra para excluir chamados das métricas.
    g = g.withColumn('qtd_flags_qualidade',reduce(lambda a,b:a+b,[F.col(c).cast('int') for c in flags]))
    return g.drop('_tem_horario','_tem_cliente','_tem_atendente','_tem_motivo','_tem_canal',
                  '_instante_abertura_us','_contatos_no_instante','_contato_anterior','_chave_motivo')


# Datasets setoriais: projeções sem joins, filtros ou recalcular métricas.
COMUNS = ['id_chamado','data_abertura','ano_mes_abertura','id_cliente','id_motivo','nome_motivo',
          'categoria','criticidade','id_canal','nome_canal','tipo_canal','regiao','faixa_idade','resolvido',
          'abandonado','recontato','qtd_contatos_anteriores_7d','transicao_canal_recontato',
          'faixa_espera','qtd_flags_qualidade']
SETORES = {
    'vw_financeiro': COMUNS + ['custo_total','qtd_registros_custo','qtd_custos_nulos','qtd_custos_negativos',
        'tempo_atendimento_segundos','nota_media','qtd_notas_validas','autosservico','flag_custo_ausente','flag_custo_invalido'],
    'vw_operacoes': COMUNS + ['id_atendente','nome_atendente','nivel_atendimento','hora_abertura_chamado',
        'hora_inicio_atendimento','hora_finalizacao_atendimento','tempo_espera_segundos',
        'tempo_ate_abandono_segundos','tempo_atendimento_segundos','tempo_total_segundos','etapa_canal',
        'qtd_etapas','flag_etapas_nao_observadas','flag_cronologia_invalida','flag_atendente_ausente'],
    'vw_experiencia_cliente': COMUNS + ['nota_media','qtd_pesquisas','qtd_notas_validas','qtd_notas_invalidas',
        'tempo_espera_segundos','tempo_atendimento_segundos','abertura_contato_anterior',
        'janela_recontato_dias','flag_pesquisa_ausente','flag_nota_invalida','flag_recontato_nao_avaliavel'],
    'vw_gestao': COMUNS + ['custo_total','nota_media','qtd_notas_validas','tempo_espera_segundos',
        'tempo_atendimento_segundos','autosservico','flag_custo_ausente'],
}


def validar_gold(gold, chamados):
    total = chamados.count()
    if gold.count() != total or gold.select('id_chamado').distinct().count() != total:
        raise ValueError('Gold alterou o grão de uma linha por chamado')
    if chamados.select('id_chamado').exceptAll(gold.select('id_chamado')).limit(1).count():
        raise ValueError('Gold perdeu IDs da Silver')
    return total


def auditar_fontes(tabelas):
    """Registros fora do universo de chamados são contabilizados, nunca incorporados via outer join."""
    ids = tabelas['tb_chamados'].select('id_chamado')
    return {nome:int(tabelas[nome].join(ids,'id_chamado','left_anti').count())
            for nome in ['tb_chamados_hora','tb_custos','tb_pesquisa_satisfacao']}


## Materialização da central, views e auditoria


In [0]:
from uuid import uuid4

faltantes = [nome for nome in CONTRATO if not spark.catalog.tableExists(f'{silver_schema}.{nome}')]
if faltantes:
    raise ValueError(f'Tabelas Silver ausentes: {faltantes}. Execute Bronze_to_Silver antes desta tarefa.')
tabelas = {nome: spark.table(f'{silver_schema}.{nome}') for nome in CONTRATO}
spark.sql(f'CREATE SCHEMA IF NOT EXISTS {gold_schema}')
# Staging específico da execução: materializa os cruzamentos sem cache/persist.
staging = f'{gold_schema}._stg_chamados_{uuid4().hex[:12]}'
try:
    (construir_gold(tabelas, janela_recontato_dias).write.format('delta')
        .mode('overwrite').option('overwriteSchema', 'true').saveAsTable(staging))
    gold = spark.table(staging)
    total = validar_gold(gold, tabelas['tb_chamados'])
    auditoria_orfaos = auditar_fontes(tabelas)
    custo_silver = (tabelas['tb_custos'].join(tabelas['tb_chamados'].select('id_chamado'), 'id_chamado', 'left_semi')
        .agg(F.sum('custo').alias('total')).first()['total'])
    custo_gold = gold.agg(F.sum('custo_total').alias('total')).first()['total']
    if custo_silver != custo_gold:
        raise ValueError(f'Custo divergente: Silver={custo_silver}, Gold={custo_gold}')
    pesquisas_silver = tabelas['tb_pesquisa_satisfacao'].join(tabelas['tb_chamados'].select('id_chamado'), 'id_chamado', 'left_semi').count()
    pesquisas_gold = gold.agg(F.sum('qtd_pesquisas')).first()[0] or 0
    if pesquisas_gold != pesquisas_silver:
        raise ValueError('Quantidade de pesquisas divergente da Silver recebida')
    (gold.write.format('delta').mode('overwrite').option('overwriteSchema', 'true')
        .saveAsTable(f'{gold_schema}.tb_chamados_central'))
finally:
    spark.sql(f'DROP TABLE IF EXISTS {staging}')

for nome, colunas in SETORES.items():
    projecao = ', '.join(f'`{c}`' for c in colunas)
    spark.sql(f'CREATE OR REPLACE VIEW {gold_schema}.{nome} AS SELECT {projecao} FROM {gold_schema}.tb_chamados_central')
central = spark.table(f'{gold_schema}.tb_chamados_central')
flags = sorted(c for c in central.columns if c.startswith('flag_'))
auditoria = central.agg(F.count('*').alias('qtd_chamados'),
    *[F.sum(F.col(c).cast('long')).alias(c) for c in flags])
for nome, qtd in auditoria_orfaos.items():
    auditoria = auditoria.withColumn(f'orfaos_{nome}', F.lit(qtd))
(auditoria.withColumn('executado_em', F.current_timestamp())
    .withColumn('qtd_pesquisas_silver', F.lit(pesquisas_silver))
    .write.format('delta').mode('overwrite').option('overwriteSchema','true')
    .saveAsTable(f'{gold_schema}.tb_auditoria_gold'))
print(f'Gold concluída: {total:,} chamados, {len(central.columns)} colunas e quatro views.')
print(f'Custo conciliado com a Silver: {custo_gold}. Pesquisas preservadas da Silver: {pesquisas_silver}.')


## Exportação completa — dentro da mesma tarefa


In [0]:
import json
from datetime import datetime, timezone
from uuid import uuid4

exportar_csv = 'true' if EXPORTAR_CSV else 'false'

OBJETOS = {
    'gold_completa': 'tb_chamados_central',
    'financeiro': 'vw_financeiro',
    'operacoes': 'vw_operacoes',
    'experiencia_cliente': 'vw_experiencia_cliente',
    'gestao': 'vw_gestao',
}

if exportar_csv == 'true':
    ausentes = [nome for nome in OBJETOS.values() if not spark.catalog.tableExists(f'{gold_schema}.{nome}')]
    if ausentes:
        raise ValueError(f'Execute Silver_to_Gold antes de exportar. Ausentes: {ausentes}')
    spark.sql(f'CREATE VOLUME IF NOT EXISTS {gold_schema}.exports')
    # Cada execução tem sua pasta. Falha não sobrescreve uma exportação anterior completa.
    execucao = datetime.now(timezone.utc).strftime('%Y%m%dT%H%M%SZ') + '_' + uuid4().hex[:8]
    pasta = f'/Volumes/{catalog}/gold/exports/{execucao}'
    dbutils.fs.mkdirs(pasta)
    total_central = spark.table(f'{gold_schema}.tb_chamados_central').count()
    manifesto = {'execucao_utc': execucao, 'separador': ';', 'encoding': 'UTF-8',
                 'decimal': '.', 'timezone': fuso_horario, 'arquivos': []}
    for nome_arquivo, objeto in OBJETOS.items():
        df = spark.table(f'{gold_schema}.{objeto}')
        total = df.count()
        if total != total_central:
            raise ValueError(f'{objeto}: {total} linhas, esperado {total_central}')
        temporario = f'{pasta}/_tmp_{nome_arquivo}'
        destino = f'{pasta}/{nome_arquivo}.csv'
        # Volume deste case permite um arquivo por dataset, sem coletar dados no driver.
        (df.coalesce(1).write.mode('overwrite').option('header', True)
            .option('sep', ';').option('encoding', 'UTF-8').option('quoteAll', True)
            .option('quote', '"').option('escape', '"').option('nullValue', '')
            .option('timestampFormat', 'yyyy-MM-dd HH:mm:ss.SSSSSS').csv(temporario))
        partes = [f.path for f in dbutils.fs.ls(temporario) if f.name.startswith('part-') and f.name.endswith('.csv')]
        if len(partes) != 1:
            raise ValueError(f'Esperava um CSV em {temporario}; encontrados {len(partes)}')
        if not dbutils.fs.mv(partes[0], destino):
            raise IOError(f'Não foi possível publicar {destino}')
        # Releitura tipada detecta quebra de aspas/linhas e confirma o total exportado.
        lido = (spark.read.schema(df.schema).option('header', True).option('sep', ';')
            .option('quote', '"').option('escape', '"').option('multiLine', True)
            .option('mode', 'FAILFAST').option('nullValue', '')
            .option('timestampFormat', 'yyyy-MM-dd HH:mm:ss.SSSSSS').csv(destino))
        if lido.count() != total:
            raise ValueError(f'CSV incompleto: {destino}')
        dbutils.fs.rm(temporario, True)
        manifesto['arquivos'].append({'arquivo': destino, 'objeto': f'{gold_schema}.{objeto}',
            'linhas': total, 'colunas': len(df.columns), 'schema': df.schema.jsonValue()})
        print(f'EXPORTADO: {destino} | {total:,} linhas | {len(df.columns)} colunas')
    # Só esta marca confirma que TODOS os arquivos da execução foram concluídos.
    dbutils.fs.put(f'{pasta}/_SUCCESS.json', json.dumps(manifesto, ensure_ascii=False, indent=2), True)
    print(f'Exportação completa. Catalog → {catalog} → gold → Volumes → exports → {execucao}')
else:
    print('Exportação CSV desativada. As tabelas Delta e views permanecem disponíveis para SELECT.')


## Exemplo de consulta


In [0]:
# Exemplos de uso direto. As views já têm os dados unificados.
# spark.sql(f"SELECT * FROM {gold_schema}.vw_financeiro")
# spark.sql(f"SELECT * FROM {gold_schema}.tb_chamados_central")
spark.sql(f"""
SELECT nome_canal, COUNT(*) AS chamados, COUNT(custo_total) AS chamados_com_custo,
       SUM(custo_total) AS custo_total,
       AVG(qtd_contatos_anteriores_7d) AS media_contatos_anteriores_7d
FROM {gold_schema}.vw_financeiro
GROUP BY nome_canal
""").show(truncate=False)


## Conferência dos resultados desta execução
Prévia da central, auditoria de qualidade e contagens das views. As prévias têm limite; os CSVs exportados contêm todas as linhas.


In [0]:
central.select('id_chamado','nome_canal','nome_motivo','custo_total','qtd_contatos_anteriores_7d','transicao_canal_recontato','faixa_espera').orderBy('id_chamado').show(10, truncate=False)

In [0]:
spark.table(f'{gold_schema}.tb_auditoria_gold').show(truncate=False)

In [0]:
for nome in ['tb_chamados_central', *SETORES]:
    df = spark.table(f'{gold_schema}.{nome}')
    print(f'{nome}: {df.count():,} linhas | {len(df.columns)} colunas')